# F1nalyze – Faza 1: Wczytanie i profilowanie danych

**Projekt:** Duży projekt klasyfikacja · **Zbiór:** F1nalyze – Formula 1 Datathon

Cel Fazy 1: wczytać i sprofilować wszystkie pliki CSV (kształt, typy, braki, rozkład klas) oraz wskazać najlepszą zmienną docelową.

> **Uwaga 1 (technika):** `train.csv` ma ~1,1 GB i 2,83 mln wierszy. Wczytujemy go porcjami (chunkami) jako tekst (`dtype=str`). Braki kodowane są jako `\N`.

> **Uwaga 2 (najważniejsza):** surowe dane to **zepsuty join** i wymagają naprawy — patrz sekcja 2. Wszystkie rozkłady klas liczymy **po naprawie**.


In [1]:
import pandas as pd, numpy as np, os
pd.set_option('display.max_columns', 60)
DATA='data'
NA=['\\N','\\n']
PLIKI=['train.csv','validation.csv','test.csv','sample_submission.csv']


## 1. Kształt plików (surowy)

| plik | wiersze (surowe) | kolumny | `position`? | zakres lat |
|---|---:|---:|---|---|
| train.csv | 2 830 101 | 55 | TAK | 1950–2013 |
| validation.csv | 353 762 | 55 | TAK | 2013–2018 |
| test.csv | 352 928 | 53 | NIE | 2018–2023 |
| sample_submission.csv | 18 | 2 | — (format) | — |

`test.csv` nie ma `position` ani `positionText_y` → układ konkursowy (Kaggle): trenujemy na `train`, oceniamy na `validation` (ma etykiety!), a `test` służy do zgłoszenia.


## 2. ⚠️ KRYTYCZNE: dane to zepsuty join (iloczyn kartezjański) — i jak to naprawić

Liczba wierszy jest **sztucznie zawyżona**. Realnych wpisów (kierowca × wyścig, klucz `resultId`) jest znacznie mniej:

| plik | wiersze | realne wpisy (`resultId`) | inflacja |
|---|---:|---:|---:|
| train | 2 830 101 | **21 643** | ~128× |
| validation | 353 762 | **1 836** | ~193× |
| test | 352 928 | **2 133** | ~165× |

W obrębie jednego `resultId` zmienne wyścigowe (`grid`, `circuitId`, `driverId`, `positionOrder`, `status`) są **stałe**, ale `position`, `points_y`, `driverStandingsId` przyjmują wiele różnych wartości — do jednego wyniku wyścigu doklejono ~130 niepowiązanych migawek klasyfikacji generalnej. **Wniosek:** surowy `position` na poziomie wiersza jest niespójny; przy uczciwych cechach pre-race jest niemożliwy do przewidzenia (ten sam wiersz cech → wiele etykiet).

**Naprawa:** zostawić tylko wiersze, gdzie `raceId_y == racerId` → dokładnie **1 wiersz na `resultId`**. Wtedy `position` staje się spójne.


In [2]:
# Demonstracja problemu i naprawy na validation (mały plik)
val=pd.read_csv(os.path.join(DATA,'validation.csv'), dtype=str, na_values=NA)
g=val.groupby('resultId')
print('wiersze:', len(val), '| realne wpisy:', g.ngroups, f'| inflacja ~{len(val)/g.ngroups:.0f}x')
print('różnych wartości na resultId (mediana):')
for col in ['positionOrder','grid','position','points_y','driverStandingsId']:
    print(f'  {col:<18}: {int(g[col].nunique().median())}  (1 = stałe w obrębie wpisu)')

# NAPRAWA
valf = val[val['raceId_y']==val['racerId']].copy()
print('\nPo naprawie: wierszy =', len(valf), '| unikalnych resultId =', valf['resultId'].nunique())
assert len(valf)==valf['resultId'].nunique(), 'powinno być 1 wiersz na resultId'


wiersze: 353762 | realne wpisy: 1836 | inflacja ~193x
różnych wartości na resultId (mediana):
  positionOrder     : 1  (1 = stałe w obrębie wpisu)
  grid              : 1  (1 = stałe w obrębie wpisu)
  position          : 18  (1 = stałe w obrębie wpisu)
  points_y          : 84  (1 = stałe w obrębie wpisu)
  driverStandingsId : 184  (1 = stałe w obrębie wpisu)

Po naprawie: wierszy = 1835 | unikalnych resultId = 1835


## 3. Podział CZASOWY (nie losowy!)

Zakresy lat są **rozłączne**: train 1950–2013, validation 2013–2018, test 2018–2023. Konsekwencje: (1) model musi generalizować w przód w czasie; (2) rozkłady i braki różnią się między plikami (inne ery F1); (3) **nie wolno** mieszać plików ani robić losowego podziału.


## 4. Typy kolumn


In [3]:
def klasyfikuj(s):
    s=s.dropna().head(50000)
    if len(s)==0: return 'pusta(100% braków)'
    n=pd.to_numeric(s, errors='coerce')
    if n.notna().mean()>0.99: return 'int' if (n.dropna()%1==0).all() else 'float'
    return 'kategoryczna/tekst'
typy=pd.Series({col:klasyfikuj(val[col]) for col in val.columns})
print(typy.value_counts())


int                   24
kategoryczna/tekst    19
pusta(100% braków)    10
float                  2
Name: count, dtype: int64


**Wynik:** 24 całkowite, 2 zmiennoprzecinkowe, 19 kategorycznych/tekstowych, **10 kolumn w 100% pustych**. Część `*Id` to faktycznie kategorie (np. `circuitId`, `constructorId`).


## 5. Braki danych


In [4]:
braki=val.isna().sum(); braki=braki[braki>0].sort_values(ascending=False)
print((braki/len(val)*100).round(1).astype(str)+' %')


fp2_date                 100.0 %
quali_date               100.0 %
sprint_time              100.0 %
sprint_date              100.0 %
quali_time               100.0 %
fp1_date                 100.0 %
fp1_time                 100.0 %
fp2_time                 100.0 %
fp3_date                 100.0 %
fp3_time                 100.0 %
timetaken_in_millisec     43.9 %
time_x                    43.9 %
position_x                17.6 %
max_speed                  4.6 %
fastestLapTime             4.6 %
fastestLap                 4.6 %
driver_num                 0.1 %
dtype: str


- **100% braków (10 kolumn — do usunięcia):** `fp1/fp2/fp3_date+time`, `quali_date+time`, `sprint_date+time`.
- **~44%:** `time_x`, `timetaken_in_millisec`. **~18–40%:** `position_x` (DNF). **~5%:** `fastestLap`, `fastestLapTime`, `max_speed`.
- Pozostałe kolumny zasadniczo kompletne.


## 6. Zmienna docelowa — rozkład PO naprawie (train)


In [5]:
def bucket(p):
    p=int(p)
    return 'wygrana(1)' if p==1 else 'podium(2-3)' if p<=3 else 'punkty(4-10)' if p<=10 else 'reszta(>10)'

use=['resultId','racerId','raceId_y','position','positionOrder']
posb={}; pob={}
for ch in pd.read_csv(os.path.join(DATA,'train.csv'), usecols=use, chunksize=500_000, dtype=str, na_values=NA):
    m=ch[ch['raceId_y']==ch['racerId']]
    for v in m['position'].dropna():      posb[bucket(v)]=posb.get(bucket(v),0)+1
    for v in m['positionOrder'].dropna(): pob[bucket(v)]=pob.get(bucket(v),0)+1
tp=sum(posb.values()); to=sum(pob.values())
print('position (klasyfikacja gen.):', {k:f'{100*v/tp:.1f}%' for k,v in sorted(posb.items())})
print('positionOrder (wynik wyścigu):', {k:f'{100*v/to:.1f}%' for k,v in sorted(pob.items())})


position (klasyfikacja gen.): {'podium(2-3)': '8.1%', 'punkty(4-10)': '26.9%', 'reszta(>10)': '60.9%', 'wygrana(1)': '4.1%'}
positionOrder (wynik wyścigu): {'podium(2-3)': '8.4%', 'punkty(4-10)': '29.1%', 'reszta(>10)': '58.3%', 'wygrana(1)': '4.2%'}


**Rozkład 4 klas (train, po naprawie):**

| klasa | `positionOrder` (wynik) | `position` (klasyfikacja) |
|---|---:|---:|
| wygrana (1) | 4,2% | 4,1% |
| podium (2–3) | 8,4% | 8,1% |
| punkty (4–10) | 29,1% | 26,9% |
| reszta (>10) | 58,3% | 60,9% |

Niezbalansowanie ~14× (umiarkowane). **Uwaga:** surowy `position` na zepsutych danych miał 108 klas i imbalance ~170 000× — to był **artefakt** zepsutego joina, nie rzeczywistość.

**Semantyka:** `position` = miejsce w klasyfikacji generalnej po wyścigu (dla niej 'wygrana' = lider mistrzostw). `positionOrder` = wynik na mecie (dla niej 'wygrana' = zwycięstwo w wyścigu).


## 7. Pułapki metodologiczne

**(A) Wyciek danych.** `positionText_y` to tekst targetu — zawsze usuwać. `points`, `position_x`, `positionOrder`, `rank` to wyniki po-wyścigowe. `points_y`/`wins` to stan PO wyścigu (skumulowany) → wolno użyć tylko w wersji **zlagowanej** (stan przed wyścigiem).

**(B) Naprawa joina** (sekcja 2) — obowiązkowa przed czymkolwiek.

**(C) Podział czasowy** (sekcja 3) — walidacja na `validation`, CV świadome czasu, bez losowego mieszania.

**(D) Kolumny-śmieci:** `url_*`, `number`, `driver_num`, `driver_code` itp. — nie są cechami; `result_driver_standing` to ID wiersza (klucz zgłoszenia).


## 8. Rekomendacja i ustalenia (wejście do Fazy 2/3)

Wybrane podejście: **naprawa danych → dwa targety (`positionOrder` i `position`) bucketowane na 4 klasy → porównanie**. Cechy bez wycieku (pre-race + historyczne zlagowane). Modele: regresja logistyczna (baseline), las losowy, gradient boosting; `class_weight='balanced'`. Metryki: macro-F1 (główna), balanced accuracy, macierz pomyłek, raport per-klasa. EDA rozbudowana. Produkt: notebook `.ipynb` (PL) + plik predykcji.

Pełna specyfikacja wykonania znajduje się w `PROMPT_dla_Claude_Code.md`.
